# Lecture 07: Moser Theorems

**Source span.** Printed pages 40-43; physical PDF pages 54-57 in `Lectures on Symplectic Geometry.pdf` according to the course source map. I checked the local PDF text for the Moser-theorem window before revising this notebook.

**Lecture goal.** This lecture turns the preparation from Lecture 6 into the Moser trick: if a path of closed nondegenerate two-forms has fixed cohomology class, choose primitives for the time derivative, solve one pointwise linear equation for a vector field, and integrate that field so the pullback of the changing form stays fixed.

The core identity is a cancellation ledger. We want `d/dt (rho_t^* omega_t) = 0`. Differentiating gives `rho_t^*(L_vt omega_t + d omega_t/dt)`. Closedness changes the Lie derivative into `d i_vt omega_t`; fixed cohomology changes `d omega_t/dt` into `d mu_t`; nondegeneracy solves `i_vt omega_t + mu_t = 0`. When all three hypotheses line up, the derivative vanishes. In the lecture's language, this is the practical test for **Equivalence of symplectic structures**: cohomologous forms joined through nondegenerate forms, with the volume constraint in dimension two visible as a never-zero area density.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")

BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json, save_matplotlib

ARTIFACT_TOPIC = "lecture-07"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
(ARTIFACT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "checks").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Artifact root: {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")

## Translation Guide

| Source idea | Computational object | Check |
| --- | --- | --- |
| symplectomorphic | a map with pullback residual zero | `A.T @ Omega1 @ A - Omega0` vanishes in a linear model |
| deformation-equivalent | a path of two-form matrices or densities | every sample remains nondegenerate |
| cohomologous forms | same de Rham class along the path | `d omega_t/dt` is exact, represented by `d mu_t` |
| volume constraint | top exterior power / area density | the sampled density stays positive, so total volume does not collapse |
| isotopic forms | same cohomology class along the path | `d omega_t/dt` is exact, represented by `d mu_t` |
| Moser vector field | solution of `i_vt omega_t = -mu_t` | pointwise linear residual is zero |
| local theorem | vector field vanishes on the submanifold | the flow fixes the core `X` |

## Library Routing

This lecture needs transparent low-dimensional models. `numpy` is enough for the pointwise Moser equation on a sampled grid, `matplotlib` makes the density, vector field, and degeneracy failure inspectable, and `networkx` captures the proof dependency chain. I avoid a heavy differential-forms package here because the lesson is the cancellation mechanism, and the exact symbolic form calculation was already handled in Lecture 6.

## Visual Storyboard

1. **Equivalence graph.** The lecture compares symplectomorphic, strongly isotopic, deformation-equivalent, and isotopic forms; an implication graph prevents these words from collapsing into one idea.
2. **Moser cancellation field.** A toy path `omega_t = (1 + 0.4 t x) dx wedge dy` has `d omega_t/dt = d(0.2 x^2 dy)`. The Moser vector field points in the `x` direction and cancels the primitive.
3. **Failure mode.** If the density crosses zero, the two-form stops being symplectic and the linear equation cannot be solved everywhere.
4. **Final ledger.** The sanity check records nondegeneracy, Moser-equation residuals, and artifact existence.

In [ ]:
# Equivalence/proof graph.
edges = [
    ("strongly isotopic", "symplectomorphic"),
    ("strongly isotopic", "isotopic"),
    ("isotopic", "deformation-equivalent"),
    ("fixed cohomology", "exact time derivative"),
    ("nondegenerate omega_t", "solve Moser equation"),
    ("closed omega_t", "Cartan cancellation"),
    ("solve Moser equation", "zero pullback derivative"),
    ("zero pullback derivative", "strong isotopy"),
]
G = nx.DiGraph(edges)
pos = {
    "strongly isotopic": (0, 1),
    "symplectomorphic": (1.8, 1.6),
    "isotopic": (1.8, 0.4),
    "deformation-equivalent": (3.7, 0.4),
    "fixed cohomology": (0, -0.9),
    "exact time derivative": (1.8, -0.9),
    "closed omega_t": (0, -1.8),
    "Cartan cancellation": (1.8, -1.8),
    "nondegenerate omega_t": (0, -2.7),
    "solve Moser equation": (1.8, -2.7),
    "zero pullback derivative": (3.7, -1.8),
    "strong isotopy": (5.3, -1.8),
}
fig, ax = plt.subplots(figsize=(9.0, 5.4))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="#64748b")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#f8fafc", edgecolors="#334155", node_size=1650)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.set_title("Moser theorem proof ledger: hypotheses feed the cancellation equation")
ax.axis("off")
proof_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "moser-proof-cancellation-graph.png")
plt.close(fig)
display_artifact(proof_path, width=820)
proof_path.relative_to(BOOK_ROOT).as_posix()

In [ ]:
# Toy Moser path: omega_t = lambda_t dx wedge dy, lambda_t = 1 + eps * t * x.
eps = 0.4
t = 0.75
xs = np.linspace(-1.8, 1.8, 33)
ys = np.linspace(-1.2, 1.2, 25)
X, Y = np.meshgrid(xs, ys)
lambda_t = 1 + eps * t * X
# mu_t = (eps/2) x^2 dy, so i_v omega_t + mu_t = 0 gives v_x = -(eps/2*x^2)/lambda_t, v_y = 0.
VX = -(eps * X**2 / 2) / lambda_t
VY = np.zeros_like(VX)
residual_dy = lambda_t * VX + eps * X**2 / 2
nondegenerate_margin = float(lambda_t.min())
moser_residual = float(np.max(np.abs(residual_dy)))

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), constrained_layout=True)
cf = axes[0].contourf(X, Y, lambda_t, levels=18, cmap="viridis")
axes[0].set_title("density of omega_t")
axes[0].set_aspect("equal")
fig.colorbar(cf, ax=axes[0], shrink=0.8)
step = (slice(None, None, 4), slice(None, None, 4))
axes[1].quiver(X[step], Y[step], VX[step], VY[step], color="#dc2626", angles="xy", scale_units="xy", scale=2.6)
axes[1].contour(X, Y, lambda_t, levels=8, colors="#94a3b8", linewidths=0.8)
axes[1].set_title("Moser vector field solving i_v omega_t + mu_t = 0")
axes[1].set_aspect("equal")
for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")
moser_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "moser-vector-field-density-cancellation.png")
plt.close(fig)
display_artifact(moser_path, width=860)
print({"nondegenerate_margin": nondegenerate_margin, "moser_residual": moser_residual})
assert nondegenerate_margin > 0
assert moser_residual < 1e-12
moser_path.relative_to(BOOK_ROOT).as_posix()

In [ ]:
# Failure mode: if lambda_t crosses zero, nondegeneracy fails and the Moser equation becomes singular.
bad_eps = 1.2
bad_t = 1.0
bad_lambda = 1 + bad_eps * bad_t * X
bad_margin = float(bad_lambda.min())
zero_crossings = int(np.sum(np.abs(bad_lambda) < 0.05))
fig, ax = plt.subplots(figsize=(6.8, 4.8))
cf = ax.contourf(X, Y, bad_lambda, levels=22, cmap="coolwarm")
ax.contour(X, Y, bad_lambda, levels=[0], colors="black", linewidths=2.0)
ax.set_title("Failure mode: density crosses zero, so omega_t is not symplectic")
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(cf, ax=ax, shrink=0.85)
failure_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "moser-nondegeneracy-failure.png")
plt.close(fig)
display_artifact(failure_path, width=680)
print({"bad_margin": bad_margin, "near_zero_samples": zero_crossings})
assert bad_margin < 0
failure_path.relative_to(BOOK_ROOT).as_posix()

## Why The Trick Works

The computation above is deliberately small. In two dimensions every two-form looks like a density times `dx wedge dy`; the Moser equation becomes one scalar cancellation. The full theorem uses the same logic pointwise: nondegeneracy of `omega_t` turns a one-form `mu_t` into a unique vector field `v_t`, and compactness supplies a flow for the whole interval.

The failure plot is just as important as the success plot. Fixed cohomology alone is not enough; the path must stay inside the open cone of nondegenerate forms. When the density crosses zero, there is no inverse two-form at that point, so the Moser equation loses its unique solution. The local theorem adds another constraint: the primitive can be chosen to vanish on `X`, so the vector field also vanishes on `X` and the resulting isotopy fixes the submanifold.

In [ ]:
source_span = {
    "course": "Lectures-on-Symplectic-Geometry",
    "lecture": 7,
    "title": "Moser Theorems",
    "printed_span": "40-43",
    "physical_pdf_span": "54-57",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "source_map_authority": "source-map.json and scripts/lsg_inventory.py",
    "source_terms_used": [
        "symplectomorphic",
        "strongly isotopic",
        "deformation-equivalent",
        "isotopic",
        "Moser trick",
        "cohomology assumption",
        "nondegeneracy assumption",
        "Moser local theorem",
    ],
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

checks = {
    "toy_path": "omega_t = (1 + eps*t*x) dx wedge dy",
    "eps": eps,
    "t": t,
    "nondegenerate_margin": nondegenerate_margin,
    "moser_equation_max_residual": moser_residual,
    "bad_eps": bad_eps,
    "bad_margin": bad_margin,
    "bad_path_detected_degeneracy": bad_margin < 0,
    "passed": nondegenerate_margin > 0 and moser_residual < 1e-12 and bad_margin < 0,
}
check_path = save_json(checks, ARTIFACT_TOPIC, "checks", "moser-equation-residuals.json")

storyboard = {
    "chapter_goal": "Make Moser's theorem visible as a cancellation equation plus a nondegeneracy gate.",
    "library_routing": [
        {"concept": "equivalence/proof route", "library": "networkx", "why": "the lecture distinguishes several equivalence notions and hypotheses"},
        {"concept": "Moser vector field", "library": "numpy + matplotlib", "why": "a sampled local model makes the pointwise linear equation inspectable"},
        {"concept": "nondegeneracy failure", "library": "matplotlib", "why": "the zero contour shows exactly where the theorem's hypothesis breaks"},
    ],
    "visual_sequence": [
        {"concept": "proof cancellation", "artifact": "artifacts/lecture-07/figures/moser-proof-cancellation-graph.png", "inspection_target": "fixed cohomology, closedness, and nondegeneracy each supply one step"},
        {"concept": "Moser vector field", "artifact": "artifacts/lecture-07/figures/moser-vector-field-density-cancellation.png", "inspection_target": "arrows solve the pointwise Moser equation"},
        {"concept": "nondegeneracy failure", "artifact": "artifacts/lecture-07/figures/moser-nondegeneracy-failure.png", "inspection_target": "zero contour marks where omega_t cannot be inverted"},
        {"concept": "residual ledger", "artifact": "artifacts/lecture-07/checks/moser-equation-residuals.json", "inspection_target": "success and failure checks are both recorded"},
    ],
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")
assert checks["passed"]
print(check_path.relative_to(BOOK_ROOT).as_posix())

## Learner Lab

Change `eps` and rerun the Moser-vector-field cell. Predict the largest value of `eps` allowed on the sampled `x` interval before the density crosses zero. Then change the primitive `mu_t`; the residual check should immediately tell you whether the proposed vector field still cancels `d omega_t/dt`.

**Takeaways.**

- Moser's theorem is a flow construction, not a classification slogan.
- The fixed cohomology hypothesis supplies the primitive `mu_t` for the time derivative of the form.
- The nondegeneracy hypothesis is what turns `mu_t` into a unique vector field.
- The compactness/global-existence condition keeps the vector field integrable for the full time interval.
- The local theorem uses the Lecture 6 homotopy formula to choose a primitive that vanishes on the submanifold, forcing the isotopy to fix it.

In [ ]:
final_artifacts = [
    "artifacts/lecture-07/figures/moser-proof-cancellation-graph.png",
    "artifacts/lecture-07/figures/moser-vector-field-density-cancellation.png",
    "artifacts/lecture-07/figures/moser-nondegeneracy-failure.png",
    "artifacts/lecture-07/checks/moser-equation-residuals.json",
    "artifacts/lecture-07/checks/source-span.json",
    "artifacts/lecture-07/checks/visual-storyboard.json",
]
residual_checks = json.loads((BOOK_ROOT / "artifacts/lecture-07/checks/moser-equation-residuals.json").read_text(encoding="utf-8"))
assert residual_checks["passed"] is True
assert residual_checks["moser_equation_max_residual"] < 1e-12
assert residual_checks["bad_path_detected_degeneracy"] is True
for relative in final_artifacts:
    path = BOOK_ROOT / relative
    assert path.exists(), f"missing artifact: {relative}"
    assert path.stat().st_size > 0, f"empty artifact: {relative}"
final_sanity = {
    "lecture": 7,
    "source_span": "printed pages 40-43; physical PDF pages 54-57",
    "artifacts": final_artifacts,
    "residual_checks": residual_checks,
    "passed": True,
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print({"checked_artifact_count": len(final_artifacts), "final_sanity": final_path.relative_to(BOOK_ROOT).as_posix()})